# Phase 4 — Feature Engineering

This notebook:

1. Loads the cleaned dataset.
2. Rebuilds the combined text feature safely.
3. Cleans the text.
4. Removes empty text rows.
5. Saves the feature-engineered dataset for model training.

Expected input:

`../dataset/processed/fake_job_postings_clean.csv`

Expected output:

`../dataset/processed/fake_job_postings_features.csv`


In [1]:
import pandas as pd
import re

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


In [2]:
DATA_PATH = Path("../dataset/processed/fake_job_postings_clean.csv")
OUTPUT_PATH = Path("../dataset/processed/fake_job_postings_features.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH.resolve()}\n"
        "Make sure the cleaned CSV is inside dataset/processed/."
    )

df = pd.read_csv(DATA_PATH)

print("Dataset path:", DATA_PATH.resolve())
print("Shape:", df.shape)
df.head()


Dataset path: /home/hayat/ml/jobshield-ai/dataset/processed/fake_job_postings_clean.csv
Shape: (17880, 18)


,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,text
0,Marketing Intern,"US, NY, New York",Marketing,NaN,"We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connect, and celebrate ...","Food52, a fast-growing, James Beard Award-winning online food community and crowd-sourced and curated recipe hub, is...",Experience with content management systems a major plus (any blogging counts!)Familiar with the Food52 editorial voi...,NaN,0,1,0,Other,Internship,NaN,NaN,Marketing,0,NaN
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"90 Seconds, the worlds Cloud Video Production Service.90 Seconds is the worlds Cloud Video Production Service enabli...",Organised - Focused - Vibrant - Awesome!Do you have a passion for customer service? Slick typing skills? Maybe Accou...,"What we expect from you:Your key responsibility will be to communicate with the client, 90 Seconds team and freelanc...",What you will get from usThrough being part of the 90 Seconds team you will gain:experience working on projects loca...,0,1,0,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,0,"Customer Service - Cloud Video Production 90 Seconds, the worlds Cloud Video Production Service.90 Seconds is the wo..."
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,"Valor Services provides Workforce Solutions that meet the needs of companies across the Private Sector, with a speci...","Our client, located in Houston, is actively seeking an experienced Commissioning Machinery Assistant that possesses ...",Implement pre-commissioning and commissioning procedures for rotary equipment.Execute all activities with subcontrac...,NaN,0,1,0,NaN,NaN,NaN,NaN,NaN,0,NaN
3,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,Our passion for improving quality of life through geography is at the heart of everything we do. Esri’s geographic ...,THE COMPANY: ESRI – Environmental Systems Research InstituteOur passion for improving quality of life through geogra...,"EDUCATION: Bachelor’s or Master’s in GIS, business administration, or a related field, or equivalent work experience...","Our culture is anything but corporate—we have a collaborative, creative environment; phone directories organized by ...",0,1,0,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,0,Account Executive - Washington DC Our passion for improving quality of life through geography is at the heart of eve...
4,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,"SpotSource Solutions LLC is a Global Human Capital Management Consulting firm headquartered in Miami, Florida. Found...","JOB TITLE: Itemization Review ManagerLOCATION: Fort Worth, TX ...","QUALIFICATIONS:RN license in the State of TexasDiploma or Bachelors of Science in Nursing, requiredPast managerial e...",Full Benefits Offered,0,1,1,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,0,Bill Review Manager SpotSource Solutions LLC is a Global Human Capital Management Consulting firm headquartered in M...


In [3]:
required_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits",
    "fraudulent",
]

missing_columns = [column for column in required_columns if column not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are available.")


All required columns are available.


## Build the combined text feature safely

Using `fillna("")` is essential. Without it, combining a string with a missing value can produce `NaN` for the entire row.


In [4]:
text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits",
]

df["text"] = (
    df[text_columns]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

print("Missing text values:", df["text"].isna().sum())
df[["text", "fraudulent"]].head()


Missing text values: 0


,text,fraudulent
0,"Marketing Intern We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connec...",0
1,"Customer Service - Cloud Video Production 90 Seconds, the worlds Cloud Video Production Service.90 Seconds is the wo...",0
2,Commissioning Machinery Assistant (CMA) Valor Services provides Workforce Solutions that meet the needs of companies...,0
3,Account Executive - Washington DC Our passion for improving quality of life through geography is at the heart of eve...,0
4,Bill Review Manager SpotSource Solutions LLC is a Global Human Capital Management Consulting firm headquartered in M...,0


In [5]:
def clean_text(text: str) -> str:
    """Normalize job-posting text for TF-IDF."""
    text = str(text).lower()

    # Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # Remove URLs
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # Keep English letters and whitespace
    text = re.sub(r"[^a-z\s]", " ", text)

    # Collapse repeated whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


In [6]:
df["clean_text"] = df["text"].apply(clean_text)

print("Missing clean_text values:", df["clean_text"].isna().sum())
print("Empty clean_text rows:", df["clean_text"].str.strip().eq("").sum())

df[["text", "clean_text", "fraudulent"]].head()


Missing clean_text values: 0
Empty clean_text rows: 0


,text,clean_text,fraudulent
0,"Marketing Intern We're Food52, and we've created a groundbreaking and award-winning cooking site. We support, connec...",marketing intern we re food and we ve created a groundbreaking and award winning cooking site we support connect and...,0
1,"Customer Service - Cloud Video Production 90 Seconds, the worlds Cloud Video Production Service.90 Seconds is the wo...",customer service cloud video production seconds the worlds cloud video production service seconds is the worlds clou...,0
2,Commissioning Machinery Assistant (CMA) Valor Services provides Workforce Solutions that meet the needs of companies...,commissioning machinery assistant cma valor services provides workforce solutions that meet the needs of companies a...,0
3,Account Executive - Washington DC Our passion for improving quality of life through geography is at the heart of eve...,account executive washington dc our passion for improving quality of life through geography is at the heart of every...,0
4,Bill Review Manager SpotSource Solutions LLC is a Global Human Capital Management Consulting firm headquartered in M...,bill review manager spotsource solutions llc is a global human capital management consulting firm headquartered in m...,0


In [7]:
before_rows = len(df)

df = df[df["clean_text"].str.strip().ne("")].copy()
df = df.drop_duplicates(subset=["clean_text", "fraudulent"]).reset_index(drop=True)

after_rows = len(df)

print("Rows before filtering:", before_rows)
print("Rows after filtering:", after_rows)
print("Rows removed:", before_rows - after_rows)


Rows before filtering: 17880
Rows after filtering: 15719
Rows removed: 2161


In [8]:
print("Target counts:")
print(df["fraudulent"].value_counts())

print("\nTarget proportions:")
print(df["fraudulent"].value_counts(normalize=True))


Target counts:
fraudulent
0    15011
1      708
Name: count, dtype: int64

Target proportions:
fraudulent
0    0.954959
1    0.045041
Name: proportion, dtype: float64


In [9]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT_PATH, index=False)

print("Saved feature-engineered dataset:")
print(OUTPUT_PATH.resolve())
print("File exists:", OUTPUT_PATH.exists())


Saved feature-engineered dataset:
/home/hayat/ml/jobshield-ai/dataset/processed/fake_job_postings_features.csv
File exists: True


In [10]:
saved_df = pd.read_csv(OUTPUT_PATH)

print("Saved shape:", saved_df.shape)
print("Missing text:", saved_df["text"].isna().sum())
print("Missing clean_text:", saved_df["clean_text"].isna().sum())

saved_df[["clean_text", "fraudulent"]].head()


Saved shape: (15719, 19)
Missing text: 0
Missing clean_text: 0


,clean_text,fraudulent
0,marketing intern we re food and we ve created a groundbreaking and award winning cooking site we support connect and...,0
1,customer service cloud video production seconds the worlds cloud video production service seconds is the worlds clou...,0
2,commissioning machinery assistant cma valor services provides workforce solutions that meet the needs of companies a...,0
3,account executive washington dc our passion for improving quality of life through geography is at the heart of every...,0
4,bill review manager spotsource solutions llc is a global human capital management consulting firm headquartered in m...,0
